# Temporal Spectral Embedding (prototype)

Spectral embedding of temporal windows via the graph Laplacian. "Spectral" here refers to the
eigenspectrum of the graph Laplacian — *not* an FFT of the time series.

This notebook is the step-1 prototype: it builds the embedding pipeline against synthetic data
before anything is wired into the real backtest. No Ridge baseline, no kNN regression, no real
RV series — only the embedding machinery.

Pipeline:

1. Take N views (each view = a W-dim vector — a slice of the recent residual series).
2. Build a sparse k-NN graph on views with Gaussian edge weights and a self-tuning bandwidth.
3. Compute the normalized graph Laplacian and its bottom-d nontrivial eigenvectors.
4. Each row of the eigenvector matrix is the embedding of the corresponding training view.
5. For a new (test) view, extend it to the embedding via weighted-nearest-neighbor Nystrom.

Once these steps work on synthetic data, the keepers graduate to `src/features/spectral_embedding.py`.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, diags, eye as sparse_eye
from scipy.sparse.linalg import eigsh
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(42)

# Locked design parameters (mirror the eventual config)
VIEW_WINDOW = 960       # length W of each view
EMBEDDING_DIM = 8       # d — bottom nontrivial Laplacian eigenvectors kept
GRAPH_K = 10            # k-NN graph neighbors for Laplacian construction

## 1. Synthetic data with known structure

Generate 3 clusters of W-dim views. Within a cluster, views share an underlying frequency
signature; across clusters they are clearly different. We expect the embedding to recover
this 3-cluster structure even though we never tell the algorithm the labels.

Each view is built as `signal + noise` where `signal` is cluster-specific:
* cluster 0: low-frequency sinusoid (slow drift)
* cluster 1: mid-frequency sinusoid + small high-frequency tremor (regime-like)
* cluster 2: white noise only (no underlying signal)

This is the textbook "can spectral embedding find clusters?" sanity check.

In [ ]:
def make_synthetic_views(n_per_cluster: int = 100, W: int = VIEW_WINDOW, noise: float = 0.5):
    """Return (views, labels) for 3-cluster synthetic data of shape (3*n, W) and (3*n,)."""
    t = np.arange(W)
    views, labels = [], []

    for c in range(3):
        for _ in range(n_per_cluster):
            phase = rng.uniform(0, 2 * np.pi)
            amp = rng.uniform(0.8, 1.2)
            if c == 0:                                                    # slow drift
                sig = amp * np.sin(2 * np.pi * t / 200 + phase)
            elif c == 1:                                                  # regime-like
                sig = amp * np.sin(2 * np.pi * t / 48 + phase)
                sig += 0.3 * amp * np.sin(2 * np.pi * t / 12 + phase)
            else:                                                         # noise-only
                sig = np.zeros(W)
            views.append(sig + noise * rng.standard_normal(W))
            labels.append(c)

    return np.asarray(views), np.asarray(labels)


views, labels = make_synthetic_views(n_per_cluster=100)
print(f"views shape: {views.shape}    (N={views.shape[0]}, W={views.shape[1]})")
print(f"labels: {np.bincount(labels)}")

In [ ]:
# Plot one representative view per cluster so it's obvious what they look like.
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
for c, ax in enumerate(axes):
    idx = np.where(labels == c)[0][0]
    ax.plot(views[idx], lw=0.8)
    ax.set_title(f"cluster {c} — first view", fontsize=9)
    ax.set_ylim(-3, 3)
axes[-1].set_xlabel("position in view")
plt.tight_layout()
plt.show()

## 2. Sparse k-NN graph with self-tuning bandwidth

For each view, find its `k_graph` nearest neighbors in Euclidean distance. Connect with Gaussian
weights `w_ij = exp(-||v_i - v_j||^2 / (2 sigma^2))`.

Bandwidth `sigma` is the **median of the k_graph-th nearest-neighbor distance** across all views
(Zelnik-Manor & Perona's self-tuning heuristic, simplified to a single global scale). This
removes the bandwidth-tuning knob and adapts to the data's natural distance scale.

The k-NN graph is asymmetric (i nearest j does not imply j nearest i). We symmetrize via
`max(W, W.T)` — a common choice that preserves all edges from either direction.

In [ ]:
def build_knn_graph(views: np.ndarray, k_graph: int):
    """Build a sparse symmetric k-NN affinity graph.

    Returns
    -------
    W : scipy.sparse.csr_matrix, shape (N, N)
        Symmetric Gaussian-weighted k-NN affinity matrix.
    sigma : float
        Bandwidth used (median k_graph-th NN distance).
    nn_model : NearestNeighbors
        Fitted NN index — reused for Nystrom extension later.
    """
    N = views.shape[0]
    nn = NearestNeighbors(n_neighbors=k_graph + 1).fit(views)
    dists, idxs = nn.kneighbors(views)        # shape (N, k_graph+1); col 0 is self

    sigma = float(np.median(dists[:, k_graph]))

    rows = np.repeat(np.arange(N), k_graph)
    cols = idxs[:, 1:].ravel()
    vals = np.exp(-dists[:, 1:].ravel() ** 2 / (2 * sigma ** 2))
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = W.maximum(W.T)                        # symmetrize
    return W, sigma, nn


W, sigma, nn_index = build_knn_graph(views, k_graph=GRAPH_K)
print(f"sigma = {sigma:.4f}")
print(f"W nnz = {W.nnz}  (dense would be {views.shape[0] ** 2})")
print(f"W is symmetric: {(W != W.T).nnz == 0}")

## 3. Normalized Laplacian + bottom-d eigenvectors

Symmetric normalized Laplacian:

$$L_{sym} = I - D^{-1/2} W D^{-1/2}$$

The bottom eigenvector (eigenvalue 0) is trivial — proportional to `sqrt(degree)` — and carries
no clustering information. We compute `d + 1` eigenvectors and drop it.

**Trick:** rather than asking `eigsh` for the smallest eigenvalues (slow with `which="SM"`),
compute the **largest** eigenvalues of `M = D^{-1/2} W D^{-1/2} = I - L_sym`. Same eigenvectors,
and `eigsh(M, which="LA")` (largest-algebraic via Lanczos) is much faster on sparse matrices.

In [ ]:
def laplacian_eigenmaps(W: csr_matrix, d: int):
    """Bottom-d nontrivial eigenvectors of the symmetric normalized Laplacian.

    Returns
    -------
    phi : np.ndarray, shape (N, d)
        The embedding coordinates for each training point.
    eigvals_L : np.ndarray, shape (d,)
        Corresponding Laplacian eigenvalues (sorted ascending).
    """
    N = W.shape[0]
    deg = np.asarray(W.sum(axis=1)).ravel()
    deg_safe = np.maximum(deg, 1e-12)
    D_inv_sqrt = diags(1.0 / np.sqrt(deg_safe))

    M = D_inv_sqrt @ W @ D_inv_sqrt           # = I - L_sym; same eigvecs, eigvals = 1 - lambda_L
    eigvals_M, eigvecs = eigsh(M, k=d + 1, which="LA")

    order = np.argsort(-eigvals_M)            # descending in M ⟺ ascending in L
    eigvals_M = eigvals_M[order]
    eigvecs = eigvecs[:, order]

    phi = eigvecs[:, 1:]                      # drop the trivial top eigvec of M
    eigvals_L = 1.0 - eigvals_M[1:]
    return phi, eigvals_L


phi_train, eigvals_L = laplacian_eigenmaps(W, d=EMBEDDING_DIM)
print(f"phi shape:   {phi_train.shape}")
print(f"Laplacian eigenvalues (small = smoother coordinate):\n{eigvals_L}")

In [ ]:
# Sanity-plot: first two embedding coordinates colored by ground-truth cluster.
# If the embedding is doing its job, the 3 clusters separate visually.

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for c in range(3):
    m = labels == c
    axes[0].scatter(phi_train[m, 0], phi_train[m, 1], s=12, alpha=0.7, label=f"cluster {c}")
    axes[1].scatter(phi_train[m, 2], phi_train[m, 3], s=12, alpha=0.7, label=f"cluster {c}")
axes[0].set_xlabel("phi_1"); axes[0].set_ylabel("phi_2"); axes[0].set_title("first 2 dims")
axes[1].set_xlabel("phi_3"); axes[1].set_ylabel("phi_4"); axes[1].set_title("dims 3-4")
axes[0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Nystrom out-of-sample extension

At test time we need to embed a new view `v_test` without rebuilding the whole graph + eigendecomp.

Pragmatic approach (Bengio et al. 2003, "Out-of-Sample Extensions for LLE, Isomap, MDS, Eigenmaps"):

1. Find `v_test`'s `k_graph` nearest training views.
2. Compute the same Gaussian kernel row.
3. The test embedding is the kernel-weighted average of those training embeddings.

This isn't the *full* Nystrom formula (which would also involve `D` and the eigenvalues), but the
weighted-kNN form is the standard cheap choice and gives the right intuition: "a test point's
embedding is interpolated from its graph neighbors."

In [ ]:
def nystrom_extend(v_test: np.ndarray, views_train: np.ndarray, phi_train: np.ndarray,
                   sigma: float, k_graph: int, nn_index: NearestNeighbors | None = None):
    """Extend a single test view to the embedding space.

    v_test     : (W,) array
    views_train: (N, W)
    phi_train  : (N, d)
    sigma      : Gaussian bandwidth from training
    k_graph    : neighbors to use (same as the one that built the graph)
    nn_index   : optional pre-fitted NearestNeighbors over views_train (cheaper for repeated calls)

    Returns
    -------
    phi_test : (d,) array — the embedding of v_test.
    """
    if nn_index is None:
        nn_index = NearestNeighbors(n_neighbors=k_graph).fit(views_train)
    dists, idx = nn_index.kneighbors(v_test[None, :], n_neighbors=k_graph)
    dists = dists.ravel()
    idx = idx.ravel()

    w = np.exp(-dists ** 2 / (2 * sigma ** 2))
    w_sum = w.sum()
    if w_sum <= 0:
        # All training neighbors are infinitely far — fall back to plain average.
        return phi_train[idx].mean(axis=0)
    w = w / w_sum
    return (w[:, None] * phi_train[idx]).sum(axis=0)

In [ ]:
# Verify Nystrom against the actual training embedding.
# Pick 30 random training points, re-embed them via Nystrom, compare to their true phi.
# This is a self-consistency check: Nystrom-extending a training point should return
# something close to its actual eigvec row.
#
# Caveat: it won't be *exactly* equal because Nystrom uses only the k_graph nearest neighbors
# (so it ignores the test point's own contribution and far-away ones).

check_idx = rng.choice(len(views), size=30, replace=False)
phi_recovered = np.stack([
    nystrom_extend(views[i], views, phi_train, sigma, GRAPH_K, nn_index=nn_index)
    for i in check_idx
])

true_phi = phi_train[check_idx]
rmse = np.sqrt(((phi_recovered - true_phi) ** 2).mean(axis=1))
print(f"Nystrom-vs-true RMSE per held-out training point:")
print(f"  min   = {rmse.min():.4f}")
print(f"  median= {np.median(rmse):.4f}")
print(f"  max   = {rmse.max():.4f}")
print(f"  embedding scale (phi std per dim) = {phi_train.std(axis=0).mean():.4f}")

In [ ]:
# Overlay Nystrom-extended NEW views on the training scatter.
# Generate 30 fresh views per cluster (drawn from the same distribution) and embed them.
# They should land near their cluster's training points.

new_views, new_labels = make_synthetic_views(n_per_cluster=30)
phi_new = np.stack([
    nystrom_extend(v, views, phi_train, sigma, GRAPH_K, nn_index=nn_index) for v in new_views
])

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["tab:blue", "tab:orange", "tab:green"]
for c in range(3):
    m = labels == c
    ax.scatter(phi_train[m, 0], phi_train[m, 1], s=14, alpha=0.4,
               c=colors[c], label=f"train cluster {c}")
    m = new_labels == c
    ax.scatter(phi_new[m, 0], phi_new[m, 1], s=40, alpha=0.95,
               edgecolors="k", linewidths=0.6, c=colors[c], label=f"new (Nystrom) cluster {c}")
ax.set_xlabel("phi_1"); ax.set_ylabel("phi_2")
ax.set_title("Nystrom-extended new views overlaid on training embedding")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

## 5. Vol-like regime test

A more realistic stress test: a synthetic 1-D series with two switching regimes (low- and high-volatility).
Slide a window of length `W` over the series to form views, embed, and color by which regime each
view ended in. If the embedding picks up the regime structure, the two colors should separate
in the scatter — even though we never told the algorithm where the regime boundaries are.

This mirrors what the real pipeline will do on HAR-OLS residuals.

In [ ]:
def make_regime_series(T: int = 6000, switch_prob: float = 0.0008,
                       sigma_low: float = 0.3, sigma_high: float = 1.5):
    """Markov-switching AR(0.7) series with two volatility regimes."""
    series = np.zeros(T)
    regime = np.zeros(T, dtype=int)
    r = 0
    for t in range(1, T):
        if rng.random() < switch_prob:
            r = 1 - r
        regime[t] = r
        eps = (sigma_low if r == 0 else sigma_high) * rng.standard_normal()
        series[t] = 0.7 * series[t - 1] + eps
    return series, regime


series, regime = make_regime_series()
print(f"series shape: {series.shape};   regime balance: {np.bincount(regime)}")

fig, ax = plt.subplots(figsize=(11, 2.5))
ax.plot(series, lw=0.4, c="k")
ax.fill_between(np.arange(len(series)), -4, 4, where=(regime == 1),
                color="tab:red", alpha=0.12, label="high-vol regime")
ax.set_ylim(-4, 4); ax.legend(loc="lower right", fontsize=8)
ax.set_title("Synthetic regime-switching series")
plt.tight_layout()
plt.show()

In [ ]:
# Form rolling views and label each view by the regime at its terminal index.
# Subsample so the eigendecomp stays fast.

W = VIEW_WINDOW
view_idx = np.arange(W, len(series), 5)            # one view every 5 steps
vol_views = np.stack([series[i - W:i] for i in view_idx])
vol_labels = regime[view_idx]
print(f"vol_views: {vol_views.shape}   label balance: {np.bincount(vol_labels)}")

W_vol, sigma_vol, nn_vol = build_knn_graph(vol_views, k_graph=GRAPH_K)
phi_vol, _ = laplacian_eigenmaps(W_vol, d=EMBEDDING_DIM)

fig, ax = plt.subplots(figsize=(7, 5))
for r, c in [(0, "tab:blue"), (1, "tab:red")]:
    m = vol_labels == r
    ax.scatter(phi_vol[m, 0], phi_vol[m, 1], s=10, alpha=0.5, c=c,
               label=("low-vol" if r == 0 else "high-vol"))
ax.set_xlabel("phi_1"); ax.set_ylabel("phi_2")
ax.set_title("Embedding of rolling views, colored by regime at window terminus")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Graduation-ready API

The three callables below are what will lift into `src/features/spectral_embedding.py`.
Together they define the public surface the model (kNN-in-embedding) will consume:

* `build_embedding(views, d, k_graph) -> SpectralBasis`
* `SpectralBasis.embed(v_test) -> (d,)` — Nystrom extension for one test view
* `SpectralBasis.embed_batch(V_test) -> (M, d)` — batched version for speed

Bundling `views_train`, `phi_train`, `sigma`, and the fitted NN index into a single
`SpectralBasis` object means the model only has to call `embed(...)` between refits.
Rebuilding is just `build_embedding(...)` again.

In [ ]:
from dataclasses import dataclass


@dataclass
class SpectralBasis:
    """Frozen training-side embedding state. Cheap to call embed() against."""
    views_train: np.ndarray            # (N, W)
    phi_train: np.ndarray              # (N, d)
    eigvals_L: np.ndarray              # (d,)
    sigma: float
    k_graph: int
    nn_index: NearestNeighbors

    def embed(self, v_test: np.ndarray) -> np.ndarray:
        return nystrom_extend(
            v_test, self.views_train, self.phi_train,
            self.sigma, self.k_graph, nn_index=self.nn_index,
        )

    def embed_batch(self, V_test: np.ndarray) -> np.ndarray:
        dists, idx = self.nn_index.kneighbors(V_test, n_neighbors=self.k_graph)
        w = np.exp(-dists ** 2 / (2 * self.sigma ** 2))
        w = w / np.clip(w.sum(axis=1, keepdims=True), 1e-12, None)
        return np.einsum("mk,mkd->md", w, self.phi_train[idx])


def build_embedding(views: np.ndarray, d: int, k_graph: int) -> SpectralBasis:
    """Full pipeline: views -> sparse k-NN graph -> Laplacian eigenmaps."""
    W, sigma, nn_index = build_knn_graph(views, k_graph=k_graph)
    phi, eigvals = laplacian_eigenmaps(W, d=d)
    return SpectralBasis(
        views_train=views, phi_train=phi, eigvals_L=eigvals,
        sigma=sigma, k_graph=k_graph, nn_index=nn_index,
    )

In [ ]:
# Smoke test of the graduation-ready API on the synthetic data.
basis = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K)
print(f"basis.phi_train.shape = {basis.phi_train.shape}")
print(f"basis.sigma           = {basis.sigma:.4f}")

# Single embed
phi_single = basis.embed(new_views[0])
print(f"single embed shape    = {phi_single.shape}")

# Batch embed
phi_batch = basis.embed_batch(new_views)
print(f"batch embed shape     = {phi_batch.shape}")
print(f"max |batch - single first row| = {np.abs(phi_batch[0] - phi_single).max():.2e}")

## Open questions / follow-ups

Things deliberately out of scope for step 1 but worth flagging before step 2:

* **True Nystrom vs. weighted-kNN.** We use the simpler weighted-kNN form. Full Nystrom
  (Coifman/Lafon-style with `D^{-1/2}` and the eigenvalues) is more accurate for points far
  from any training neighbor but rarely needed when test views are drawn from the training
  distribution.
* **Symmetrization choice.** We use `max(W, W.T)`. Alternatives: `(W + W.T) / 2`, or mutual k-NN
  (`W.minimum(W.T)`). Worth A/B-ing once a baseline metric exists.
* **Sigma policy.** Median of `k_graph`-th NN distance gives one global scale. Per-pair local
  scaling (`sigma_i * sigma_j`) is the Zelnik-Manor & Perona original and often does better
  on heterogeneous-density data.
* **Compute envelope at N≈24 k.** `eigsh(M, k=9, which="LA")` on a sparse N=24 k, k_graph=10 matrix
  is ~seconds on a CPU. Acceptable at `refit_frequency=240`. Worth re-measuring before lifting.
* **Determinism.** Lanczos has a random starting vector. We will need to pass `v0=` / set the
  numpy seed in the lifted version for reproducible eigenvectors (matters because subsequent
  rebuilds compare).
* **Sign + axis ambiguity.** Eigenvectors are only defined up to a sign flip. If we ever compare
  embeddings across two refits, we will need to align them (Procrustes or sign-matching).